# Arcane Splitter – Google Colab

Slice grid images (3×4 = 12 cells), auto-crop white borders, generate Midjourney prompts (OpenAI Vision), upload slices to **WordPress** (URL + prompt format), run **bulk TTAPI** (Midjourney) generation, and upload results to **Google Drive**.

**Flow:**
1. Install deps, upload `arcane_splitter.py`, set API keys (OpenAI, Etsy optional).
2. **Step 4**: Upload image(s) or Etsy URL → **source_images**.
3. **Step 4b**: Review images (with indices), choose which to keep or delete → **selected_images**.
4. **Step 5**: Slice selected images (and optionally analyze with OpenAI).
5. **WordPress** (optional): upload slices, get prompts as `https://.../image.png Full prompt --ar 3:4 --v 6.1 --s 0`.
6. **Bulk TTAPI**: paste prompts, generate images with Midjourney.
7. **Google Drive**: upload slices or TTAPI-generated images to a folder.

## 1. Install dependencies

In [ ]:
!pip install -q Pillow numpy requests openai

## 2. Upload arcane_splitter.py

Upload the `arcane_splitter.py` file from the `colab/` folder of this repo, or clone the repo and use the file from there.

In [ ]:
# Option A: Upload the file (run this cell, then use the file picker)
from google.colab import files
uploaded = files.upload()  # Choose arcane_splitter.py

# Option B: Clone the repo (uncomment if you prefer)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO/colab

## 3. Set API keys

In [ ]:
import os

# OpenAI (required for "Analyze" step)
OPENAI_API_KEY = ""  # Paste your key, or use: getpass.getpass("OpenAI key: ")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Etsy (optional – only if you use an Etsy listing URL)
ETSY_API_KEY = ""  # Get one at https://www.etsy.com/developers/
if ETSY_API_KEY:
    os.environ["ETSY_API_KEY"] = ETSY_API_KEY

## 4. Load the module and choose input (images to slice)

This step builds **source_images** (a list). For a single upload it has one item; for Etsy it has one item per listing image.

In [ ]:
from arcane_splitter import (
    slice_grid_image,
    run_arcane_splitter,
    save_slices_as_zip,
    listing_id_from_url,
    fetch_etsy_listing_images,
)
from PIL import Image
import io

# --- Choose ONE input method ---

# Option A: Upload one or more grid images (run cell, then pick files)
from google.colab import files
uploaded = files.upload()
# source_images = list of paths (one per file)
source_images = list(uploaded.keys())

# Option B: Use an Etsy listing URL (set ETSY_API_KEY above) – fetches all listing images
# etsy_url = "https://www.etsy.com/listing/1206601867/..."
# listing_id = listing_id_from_url(etsy_url) or etsy_url.split(":")[-1].strip()
# images_bytes = fetch_etsy_listing_images(listing_id)
# source_images = [Image.open(io.BytesIO(ib)).convert("RGB") for ib in images_bytes]

print(f"Loaded {len(source_images)} image(s). Run the next cell to review and choose which to slice.")

## 4b. Review and choose images to slice (delete ones you don’t want)

Images are shown below with their **index**. Set **KEEP_INDICES** (e.g. `[0, 1, 3]`) to keep only those, or **DELETE_INDICES** (e.g. `[2, 4]`) to remove those. Then run the cell. Only **selected_images** will be sliced in step 5.

In [ ]:
from IPython.display import display, HTML
from pathlib import Path

# Show each source image with its index (thumbnail)
def show_source_images(images, max_size=300):
    for i, img in enumerate(images):
        if isinstance(img, (str, Path)):
            img = Image.open(img).convert("RGB")
        # else assume PIL Image
        img = img.copy()
        img.thumbnail((max_size, max_size))
        display(HTML(f"<b>Index {i}</b>"))
        display(img)

show_source_images(source_images)
print(f"Total: {len(source_images)} image(s). Indices 0 to {len(source_images)-1}.")

# --- Choose which to keep (edit one of these) ---
# Option 1: Keep only these indices (e.g. [0, 1, 3] → slice images 0, 1, 3 only)
KEEP_INDICES = None  # e.g. [0, 1, 3] or None to keep all

# Option 2: Delete these indices (e.g. [2, 4] → remove images 2 and 4)
DELETE_INDICES = []  # e.g. [2, 4] or [] to delete none

# Build selected list
if KEEP_INDICES is not None:
    selected_indices = [i for i in KEEP_INDICES if 0 <= i < len(source_images)]
else:
    selected_indices = [i for i in range(len(source_images)) if i not in DELETE_INDICES]

selected_images = [source_images[i] for i in selected_indices]
print(f"Selected {len(selected_images)} image(s) to slice (indices {selected_indices}).")

## 5. Slice the grid (and optionally analyze with OpenAI)

In [ ]:
# Uses selected_images from step 4b (or run 4b with KEEP_INDICES=None, DELETE_INDICES=[] to use all)
slices, results = run_arcane_splitter(
    selected_images,
    rows=3,
    cols=4,
    auto_crop=True,
    openai_api_key=os.environ.get("OPENAI_API_KEY") or OPENAI_API_KEY or None,
    main_topic="",  # Optional theme, e.g. "botanical watercolor"
    analyze=bool(os.environ.get("OPENAI_API_KEY") or OPENAI_API_KEY),
)

print(f"Slices: {len(slices)}")
if results:
    print(f"Analyzed: {len(results)}")

## 6. Preview slices (optional)

In [ ]:
from IPython.display import display

for i, s in enumerate(slices[:6]):  # show first 6
    display(s["image_pil"])

## 7. Download slices as ZIP

In [ ]:
zip_path = save_slices_as_zip(slices, "arcane_slices.zip")
from google.colab import files
files.download(zip_path)
print("Downloaded:", zip_path)

## 8. Print or copy prompts (if you ran analysis)

In [ ]:
if results:
    for i, r in enumerate(results):
        print(f"--- Image {i+1}: {r.get('name', '')} ---")
        print(r.get("prompt", "")[:300] + "..." if len(r.get("prompt", "")) > 300 else r.get("prompt", ""))
        print()
    all_prompts = "\n\n".join(r.get("prompt", "") for r in results)
    print("\n(Full prompts are in the variable 'results' – copy as needed.)")
else:
    print("No analysis results. Set OPENAI_API_KEY and run with analyze=True.")

## 9. WordPress upload (optional)

Upload each slice to your WordPress site, then format prompts as: **URL + prompt** (one line per image). Use these lines for Midjourney style references or bulk generation.

In [ ]:
# WordPress credentials (Application Password from Users → Profile → Application Passwords)
WP_URL = "https://your-site.hostingersite.com"  # No trailing slash, e.g. https://gold-stingray-884517.hostingersite.com
WP_USERNAME = ""
WP_APP_PASSWORD = ""

if WP_URL and WP_USERNAME and WP_APP_PASSWORD and "slices" in dir() and "results" in dir() and len(slices) == len(results):
    from arcane_splitter import upload_slices_to_wordpress, format_prompts_with_urls

    wp_urls = upload_slices_to_wordpress(slices, WP_URL, WP_USERNAME, WP_APP_PASSWORD)
    prompts_with_urls = format_prompts_with_urls(wp_urls, results)

    # Print one line per image: "https://.../image.png Full prompt text --ar 3:4 --v 6.1 --s 0"
    for line in prompts_with_urls:
        print(line)
        print()

    # Save for next step (Bulk TTAPI)
    BULK_PROMPTS = prompts_with_urls
else:
    if not (WP_URL and WP_USERNAME and WP_APP_PASSWORD):
        print("Set WP_URL, WP_USERNAME, WP_APP_PASSWORD above.")
    else:
        print("Run the slice + analyze cells first so 'slices' and 'results' exist and match length.")
    BULK_PROMPTS = [r.get("prompt", "") for r in results] if "results" in dir() and results else []

## 10. Bulk prompt import – generate images with TTAPI (Midjourney)

Paste prompts (one per line). Each line can be **URL + prompt** (e.g. from WordPress step) or plain prompt. TTAPI will submit each to Midjourney and poll until complete.

In [ ]:
# TTAPI (Midjourney) – get key from https://ttapi.io or your dashboard
TTAPI_API_KEY = ""
TTAPI_DOMAIN = "https://api.ttapi.io"  # or your custom domain

# Prompts: use BULK_PROMPTS from WordPress step above, or paste your own (one per line)
BULK_PROMPTS = BULK_PROMPTS if 'BULK_PROMPTS' in dir() else [
    "https://example.com/image.png A majestic black raven... --ar 3:4 --v 6.1 --s 0",
    # Add more lines or paste below
]

# Optional: paste multi-line block and split
# BULK_PROMPTS = """paste here""".strip().split("\n")
# BULK_PROMPTS = [p.strip() for p in BULK_PROMPTS if p.strip()]

## 11. Google Drive upload

Upload generated images (from TTAPI) or your slices to a Google Drive folder. Set **GOOGLE_DRIVE_PARENT_FOLDER_ID** (parent folder, from the Drive URL) and **DRIVE_OUTPUT_FOLDER_NAME** (name of the folder where results will be saved; it is created under the parent). You need OAuth: Client ID, Client Secret, and Refresh Token (see your project’s README or Google Cloud Console).

In [ ]:
# Google Drive: parent folder (ID from Drive URL) and name of folder where results will be saved
GOOGLE_DRIVE_CLIENT_ID = ""
GOOGLE_DRIVE_CLIENT_SECRET = ""
GOOGLE_DRIVE_REFRESH_TOKEN = ""
GOOGLE_DRIVE_PARENT_FOLDER_ID = ""   # Parent folder ID (from Drive URL). Files go into a new subfolder under this.
DRIVE_OUTPUT_FOLDER_NAME = "ArcaneSplitter-Output"   # Name of the folder to create under the parent (you choose this).

# Upload images to Google Drive
# Option A: Upload TTAPI-generated images (download from imageUrl first)
# Option B: Upload slices from this notebook (slices variable)
import requests
from arcane_splitter import google_drive_upload_images, google_drive_refresh_token

def _drive_create_folder(access_token, parent_folder_id, folder_name):
    """Create a new folder in Drive under parent. Returns the new folder id."""
    r = requests.post(
        "https://www.googleapis.com/drive/v3/files",
        headers={"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"},
        json={"name": folder_name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_folder_id]},
        timeout=30)
    r.raise_for_status()
    return r.json()["id"]

try:
    _ = ttapi_results
except NameError:
    ttapi_results = []
try:
    _ = slices
except NameError:
    slices = []

# Choose source: "ttapi" (download from ttapi_results) or "slices" (use local slices)
UPLOAD_SOURCE = "slices"  # or "ttapi"

if UPLOAD_SOURCE == "ttapi" and ttapi_results:
    # Download images from TTAPI result URLs
    image_bytes_list = []
    filenames = []
    for i, r in enumerate(ttapi_results):
        url = r.get("imageUrl") or r.get("image_url")
        if not url:
            continue
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        image_bytes_list.append(resp.content)
        filenames.append(f"mj_{i+1}.png")
    images_to_upload = image_bytes_list
    names = filenames
elif UPLOAD_SOURCE == "slices" and slices:
    images_to_upload = [s["image_pil"] for s in slices]
    names = [f"slice_{i+1}_r{s['row']+1}c{s['col']+1}.png" for i, s in enumerate(slices)]
else:
    images_to_upload = []
    names = []

if images_to_upload and all([GOOGLE_DRIVE_CLIENT_ID, GOOGLE_DRIVE_CLIENT_SECRET, GOOGLE_DRIVE_REFRESH_TOKEN, GOOGLE_DRIVE_PARENT_FOLDER_ID]):
    output_folder_name = (DRIVE_OUTPUT_FOLDER_NAME or "ArcaneSplitter-Output").strip().replace("/", "_").replace("\\", "_") or "ArcaneSplitter-Output"
    token = google_drive_refresh_token(GOOGLE_DRIVE_CLIENT_ID, GOOGLE_DRIVE_CLIENT_SECRET, GOOGLE_DRIVE_REFRESH_TOKEN)
    folder_id = _drive_create_folder(token, GOOGLE_DRIVE_PARENT_FOLDER_ID.strip(), output_folder_name)
    drive_links = google_drive_upload_images(
        GOOGLE_DRIVE_CLIENT_ID,
        GOOGLE_DRIVE_CLIENT_SECRET,
        GOOGLE_DRIVE_REFRESH_TOKEN,
        folder_id,
        images_to_upload,
        filenames=names,
    )
    print(f"Saved {len(drive_links)} file(s) to folder: {output_folder_name}")
    for d in drive_links:
        print(d["name"], d["webViewLink"])
else:
    print("Set Google Drive credentials (including GOOGLE_DRIVE_PARENT_FOLDER_ID and DRIVE_OUTPUT_FOLDER_NAME), UPLOAD_SOURCE, and ensure ttapi_results/slices exist.")